# Hearo - Fine-tune YAMNet on ESC-50

Train a custom **on-device** sound classifier for Hearo and export it to TensorFlow.js.

**What this does**
1. Downloads the [ESC-50](https://github.com/karolpiczak/ESC-50) dataset (2,000 labeled environmental sounds, 50 classes)
2. Uses Google's **YAMNet** as a frozen feature extractor (1024-dim embeddings)
3. Trains a small classifier head on top
4. Exports everything to **TensorFlow.js** so it runs in the browser - free, offline, on-device
5. Zips the model + labels for you to download and drop into `public/models/`

**How to run**
- `Runtime -> Change runtime type -> GPU` (optional, speeds up embedding extraction)
- Run the **Setup** cell, then when Colab asks, **Runtime -> Restart session**
- Then `Runtime -> Run all`

The whole run takes ~10-20 minutes (most of it is the 600 MB dataset download + embedding extraction).


## Setup - install pinned dependencies

We pin TensorFlow 2.15 / tensorflowjs 4.17 because newer Colab images ship Keras 3,
which the TF.js converter doesn't support yet. **After this cell finishes, restart the
runtime** (Runtime -> Restart session), then run the rest of the notebook.


In [ ]:
!pip install -q tensorflow==2.15.0 tensorflow_hub==0.16.1 tensorflowjs==4.17.0 \
    librosa==0.10.1 soundfile pandas tqdm scikit-learn

print("\nDependencies installed.")
print(">>> Now do: Runtime -> Restart session, then Run all. <<<")


## Step 1 - Download ESC-50 (~600 MB)

In [ ]:
import os, urllib.request, zipfile

if not os.path.isdir('ESC-50-master'):
    print('Downloading ESC-50 (~600 MB, one time)...')
    urllib.request.urlretrieve(
        'https://github.com/karolpiczak/ESC-50/archive/master.zip', 'esc50.zip')
    with zipfile.ZipFile('esc50.zip') as z:
        z.extractall('.')
    print('Extracted.')
else:
    print('ESC-50 already downloaded.')

AUDIO_DIR = 'ESC-50-master/audio'
META_CSV  = 'ESC-50-master/meta/esc50.csv'
print('Audio files:', len(os.listdir(AUDIO_DIR)))


## Step 2 - Load YAMNet (feature extractor)

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np

print('TensorFlow', tf.__version__)
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')
print('YAMNet loaded - outputs 1024-dim embeddings per 0.48s frame.')


## Step 3 - Read labels

ESC-50 has 50 classes. `target` (0-49) is the class index; `category` is the name.


In [ ]:
import pandas as pd

meta = pd.read_csv(META_CSV)
class_map = meta[['target', 'category']].drop_duplicates().sort_values('target')
LABELS = class_map['category'].tolist()          # index position == target id
NUM_CLASSES = len(LABELS)

print(NUM_CLASSES, 'classes:')
print(LABELS)


## Step 4 - Extract YAMNet embeddings for every clip

Each 5-second clip becomes ~10 frame-embeddings of size 1024. We train on the
frame-level embeddings (more samples = better generalization).

This is the slow part - a few minutes.


In [ ]:
import librosa
from tqdm.auto import tqdm

def load_wav_16k_mono(path):
    # YAMNet expects 16 kHz mono float32 in [-1, 1]
    wav, _ = librosa.load(path, sr=16000, mono=True)
    return wav.astype(np.float32)

X_list, y_list = [], []
for _, row in tqdm(meta.iterrows(), total=len(meta), desc='Extracting embeddings'):
    wav = load_wav_16k_mono(os.path.join(AUDIO_DIR, row['filename']))
    _, embeddings, _ = yamnet(wav)          # embeddings: [frames, 1024]
    emb = embeddings.numpy()
    X_list.append(emb)
    y_list.append(np.full(emb.shape[0], row['target'], dtype=np.int64))

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)
print('Embeddings:', X.shape, '| labels:', y.shape)


## Step 5 - Train the classifier head

A small dense network on top of the frozen YAMNet embeddings.


In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

head = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1024,), name='embedding'),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', name='predictions'),
], name='hearo_head')

head.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
             loss='sparse_categorical_crossentropy',
             metrics=['accuracy'])
head.summary()

es = tf.keras.callbacks.EarlyStopping(
    patience=6, restore_best_weights=True, monitor='val_accuracy')

head.fit(X_tr, y_tr, validation_data=(X_te, y_te),
         epochs=60, batch_size=64, callbacks=[es])


## Step 6 - Evaluate

Frame-level accuracy. Real-world (clip-level, averaged over frames) accuracy is
usually a few points higher.


In [ ]:
loss, acc = head.evaluate(X_te, y_te, verbose=0)
print(f'Frame-level test accuracy: {acc:.1%}')

# Per-class report
from sklearn.metrics import classification_report
y_pred = head.predict(X_te, verbose=0).argmax(axis=1)
print(classification_report(y_te, y_pred, target_names=LABELS, zero_division=0))


## Step 7 - Export to TensorFlow.js

The head is a plain dense network, so it converts cleanly. At inference the app will:
YAMNet -> embeddings `[frames, 1024]` -> this head -> `[frames, 50]` -> mean over frames.


In [ ]:
import tensorflowjs as tfjs

OUT_DIR = 'hearo_sound_model'
tfjs.converters.save_keras_model(head, OUT_DIR)
print('TF.js model written to', OUT_DIR)
print(os.listdir(OUT_DIR))


## Step 8 - Save labels + Hearo category mapping

`labels.json` - class names in index order.
`esc50_to_hearo.json` - maps ESC-50 classes to Hearo alert categories (the rest are
treated as non-alert background, same as "Speech" in the current app).


In [ ]:
import json

with open(os.path.join(OUT_DIR, 'labels.json'), 'w') as f:
    json.dump(LABELS, f, indent=2)

# ESC-50 class -> Hearo alert category (must match SoundCategories keys in App.jsx)
esc50_to_hearo = {
    'crying_baby':    'baby_cry',
    'dog':            'dog_bark',
    'glass_breaking': 'glass_break',
    'siren':          'siren',
    'car_horn':       'car_horn',
    'clock_alarm':    'alarm',
    'door_wood_knock':'knock',
    'church_bells':   'doorbell',
}
with open(os.path.join(OUT_DIR, 'esc50_to_hearo.json'), 'w') as f:
    json.dump(esc50_to_hearo, f, indent=2)

print('Saved labels.json and esc50_to_hearo.json')
print('Mapped alert classes:', list(esc50_to_hearo.keys()))


## Step 9 - Download the model bundle

In [ ]:
import shutil
shutil.make_archive('hearo_sound_model', 'zip', OUT_DIR)

from google.colab import files
files.download('hearo_sound_model.zip')
print('Downloaded hearo_sound_model.zip')


## Step 10 - Integrate into the Hearo app

1. Unzip `hearo_sound_model.zip` into the repo at **`public/models/hearo/`**:
   ```
   public/models/hearo/
     model.json
     group1-shard1of1.bin   (weights)
     labels.json
     esc50_to_hearo.json
   ```
2. Come back to the chat and tell me the model is in place - I'll wire `App.jsx` to:
   - load this head model alongside YAMNet,
   - run YAMNet to get **embeddings** (output index 1) instead of the 521-class scores,
   - run the head on those embeddings and map results via `esc50_to_hearo.json`,
   - add a Settings toggle to switch between the **stock YAMNet (521 classes)** and your
     **custom ESC-50 model**, so you can A/B them.

That keeps everything on-device and free, and lets you compare the two models directly.
